# TPU 空き確認プローブ

TPU VM が起動するか・デバイスが見えるかだけを見る。計算は 1024x1024 の 1 回だけ。

In [ ]:
import os, platform, sys, time
print("time      :", time.strftime("%Y-%m-%d %H:%M:%S"))
print("platform  :", platform.platform())
print("python    :", sys.version.split()[0])
print("cpu_count :", os.cpu_count())
try:
    print("meminfo   :", open("/proc/meminfo").readline().strip())
except Exception as e:
    print("meminfo   :", e)
print("TPU_NAME  :", os.environ.get("TPU_NAME"))
print("TPU_WORKER:", os.environ.get("TPU_WORKER_HOSTNAMES"))
print("XRT_CFG   :", os.environ.get("XRT_TPU_CONFIG"))
import glob
print("accel devs:", glob.glob("/dev/accel*") + glob.glob("/dev/vfio/*")[:4])


In [ ]:
import importlib.util as u
for m in ("torch", "torch_xla", "jax", "jaxlib", "tensorflow", "libtpu"):
    print(f"{m:12s}", bool(u.find_spec(m)))


In [ ]:
# JAX 経路（Kaggle TPU イメージの標準）
try:
    import jax, jax.numpy as jnp
    print("jax", jax.__version__)
    devs = jax.devices()
    print("devices", devs)
    x = jnp.ones((1024, 1024), dtype=jnp.float32)
    y = float((x @ x).sum())
    print("matmul sum", y, "expected", 1024 * 1024 * 1024)
    print("JAX_OK", abs(y - 1024**3) < 1)
except Exception as e:
    print("JAX_FAIL", type(e).__name__, e)


In [ ]:
# PyTorch/XLA 経路
try:
    import torch
    import torch_xla.core.xla_model as xm
    print("torch", torch.__version__)
    d = xm.xla_device()
    print("xla_device", d)
    a = torch.randn(1024, 1024, device=d)
    s = float((a @ a).sum().cpu())
    print("xla matmul sum", s)
    print("TORCH_XLA_OK", True)
except Exception as e:
    print("TORCH_XLA_FAIL", type(e).__name__, e)


In [ ]:
# 結果を成果物にも残す（ログが取れない場合の保険）
try:
    with open("/kaggle/working/probe.txt", "w") as f:
        f.write("ok\n")
    print("wrote /kaggle/working/probe.txt")
except Exception as e:
    print("write fail", e)
print("PROBE_COMPLETE")
